# Generate amorphous LSU networks

This notebook demonstrates `generate_lsu_network`, which implements the
Wooten-Winer-Weaire simulated annealing algorithm of Sellers et al.
(*Nat. Commun.* **8**, 14439, 2017) on a periodic 3-regular graph.

Documentation: see the `claude_context/` folder.

**Inputs**: target LSU (`lsu_degree_12` or `lsu_degree_22`), one of
(`num_vertices` or `num_rods`), `bounds_microns`.

**Output**: NumPy array `(R, 6)` where each row is `[x1, y1, z1, x2, y2, z2]` —
directly usable in the `create_permittivity_grid_penlike` pipeline. With the
default `pbc_duplicate_boundary_rods=True`, `R` equals the unique-edge count
plus the number of edges crossing box faces (each rendered twice, once from
each canonical-box endpoint), matching the Sellers reference file convention.

In [1]:
import numpy as np
import os
import lsu_network as lsu
import jax 

print('JAX available:', lsu.HAS_JAX)
print("Devices:", jax.devices())

JAX available: True
Devices: [CudaDevice(id=0)]


## Reproduce the example: 1000 vertices / 1500 unique edges, periodicity 11.44 µm

The reference example (`Example/lsu_example_ends.txt`) has Φ_12 ≈ 0.99 and
Φ_22 ≈ 0.89, **N=1000 vertices, E=1500 unique edges**, periodicity 11.44 µm,
mean rod length 0.8 µm. The 1653 lines in that file include 153 PBC-image
duplicates of edges crossing box faces — required so that
`create_permittivity_grid_penlike` (which draws each rod as a literal cylinder
and does not apply PBC) produces a periodic permittivity grid.

At full scale this needs ~50,000 WWW iterations. With JAX it's tractable
(JIT-compiled energy + autodiff gradient); without JAX, plan to run
overnight or reduce iterations.

In [2]:
# Adjust n_www_iterations down if you want a quicker (less converged) run.
# num_vertices=1000 matches the Sellers reference topology (N=1000 / E=1500).
# With pbc_duplicate_boundary_rods=True (default) each face-crossing edge is
# emitted twice, so len(rods) = 1500 + (boundary duplicates) - close to but
# not exactly 1653 (depends on the specific edge geometry).
#
# Notes on relaxation kwargs (Vink/Mousseau-Barkema scheme, Sellers refs [13,14]):
#   - local_shell_depth=4 restricts the post-SW L-BFGS to vertices within 4
#     graph-edge hops of the move; out-of-shell vertices are held fixed.
#     This is what eliminates the corner/face void clustering observed in
#     earlier runs (full-N L-BFGS lets vertices anywhere in the cell drift
#     toward each other under the bonded-only Sellers energy).
#   - relax_global_every is deprecated (default 0). The previous fixed-
#     schedule full-N polish re-introduced the same drift.
#   - global_fallback_threshold (default float('inf'), off) opts in to a
#     Vink/MB-style fallback: only fires when local relax fails to settle
#     and dE > threshold. Set e.g. 10.0 to enable; tune up if it fires too
#     often, down if too rarely.
rods = lsu.generate_lsu_network(
    lsu_degree_22=0.89,            # could also use lsu_degree_12=0.99
    num_vertices=1000,
    bounds_microns=11.44,
    edge_length=0.8,
    n_www_iterations=100_000,
    initial_temperature=1.0,
    final_temperature=1e-5,
    target_tolerance=0.01,
    check_lsu_every=500,
    relax_local_iters=120,
    relax_global_iters=200,        # only used if the fallback gate fires
    # global_fallback_threshold=10.0,  # uncomment to enable Vink/MB fallback
    local_shell_depth=4,           # Vink/MB local-shell relax (Sellers refs [13,14])
    seed=369,
    use_jax=True,
    verbose=True,
    energy_weights={'alpha':50, 'beta':5, 'gamma':1, 'delta':0.5}
)
print('shape:', rods.shape)

[gen] N=1000 vertices, E=1500 rods, box=[11.44, 11.44, 11.44], d0=0.8, target phi_22=0.89, jax=on, jaxopt=off
[gen] BM seed: rod length mean=1.022, std=0.351, min=0.561, max=4.263
[gen] initial relax: E=3590
[WWW it=   500] T=0.9442  E=2871  phi_22=0.5621  acc=38.80%  fb=0.00%  elapsed=43.8s
[WWW it=  1000] T=0.8914  E=2477  phi_22=0.5856  acc=34.10%  fb=0.00%  elapsed=84.2s
[WWW it=  1500] T=0.8415  E=2179  phi_22=0.6019  acc=31.40%  fb=0.00%  elapsed=121.3s
[WWW it=  2000] T=0.7944  E=1980  phi_22=0.6119  acc=29.20%  fb=0.00%  elapsed=158.6s
[WWW it=  2500] T=0.75  E=1812  phi_22=0.6266  acc=27.16%  fb=0.00%  elapsed=194.8s
[WWW it=  3000] T=0.708  E=1645  phi_22=0.6327  acc=25.97%  fb=0.00%  elapsed=229.3s
[WWW it=  3500] T=0.6684  E=1537  phi_22=0.6447  acc=24.51%  fb=0.00%  elapsed=262.2s
[WWW it=  4000] T=0.631  E=1437  phi_22=0.6478  acc=23.35%  fb=0.00%  elapsed=295.5s
[WWW it=  4500] T=0.5957  E=1358  phi_22=0.6595  acc=22.29%  fb=0.00%  elapsed=329.6s
[WWW it=  5000] T=0.5624

## Save the output

The 6-column form (x1,y1,z1,x2,y2,z2) is directly compatible with
`np.loadtxt` as used by the rest of the pipeline. The 7-column form below
(with a 1-based index column) matches `Example/lsu_example_ends.txt`.

In [3]:
os.makedirs('./Example', exist_ok=True)

# 6-column compatible with create_permittivity_grid_penlike
np.savetxt('./Example/lsu_generated_5.txt', rods,
           fmt=' '.join(['%.14g'] * 6), delimiter='\t')

# # 7-column with index, matching Example/lsu_example_ends.txt
# indexed = np.column_stack([np.arange(1, len(rods) + 1), rods])
# np.savetxt('./Example/lsu_generated_indexed.txt', indexed,
#            fmt='%d\t' + '\t'.join(['%.14g'] * 6))

print('saved', rods.shape[0], 'rods')

saved 1645 rods


## Verify the result

Quick checks: connectivity (rods belong to one connected network), edge length
distribution, and final LSU values.

In [4]:
BOX = 11.44  # Must match bounds_microns above for the stats below to be meaningful.
p1 = rods[:, :3]
p2 = rods[:, 3:]
lengths = np.linalg.norm(p2 - p1, axis=1)

# 1) Rod-length distribution. Reference example (1653 rods, BOX=11.44):
#    mean=0.800 std=0.029  q5=0.752 med=0.801 q95=0.846 min=0.667 max=0.884
qs = np.quantile(lengths, [0.0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0])
print(f'rod count : {len(rods)}')
print(f'lengths   : mean={lengths.mean():.3f} std={lengths.std():.3f}')
print(f'  quartiles  min={qs[0]:.3f}  5%={qs[1]:.3f}  Q1={qs[2]:.3f}  '
      f'med={qs[3]:.3f}  Q3={qs[4]:.3f}  95%={qs[5]:.3f}  max={qs[6]:.3f}')
print(f'  ref target  mean=0.800 std=0.029 (reach with enough WWW iters)')

# 2) Spatial-coverage check - tile the canonical box into 1 µm cells and
# count cells with no vertex. Reference example: 54.5% empty (Poisson at
# density 0.74 verts/µm³ would naturally give ~44% empty). The thing the
# old configuration-model seed got wrong was *clusters* of empty cells -
# multi-µm voids. The Poisson-disk seed used now should give a roughly
# Poisson-like empty-cell pattern with no large connected void region.
half = BOX / 2.0
verts = np.vstack([p1, p2])
verts_canon = verts - BOX * np.round(verts / BOX)
n_cells = int(np.ceil(BOX))
edges_grid = np.linspace(-half, half, n_cells + 1)
H, _ = np.histogramdd(verts_canon, bins=(edges_grid, edges_grid, edges_grid))
empty = int(np.sum(H == 0))
print(f'1 µm³ vertex coverage: {H.size} cells, {empty} empty '
      f'({empty / H.size:.1%})  (reference: 54.5%)')

try:
    from scipy.ndimage import label
    labeled, n_components = label(H == 0)
    sizes = sorted((int((labeled == c).sum()) for c in range(1, n_components + 1)),
                   reverse=True)
    print(f'  largest empty clusters: {sizes[:5]}  '
          f'(big single cluster is normal at this density due to PBC '
          f'percolation; what was *wrong* before was a big cluster on a '
          f'box face)')
except ImportError:
    pass

print(f'box span   : x [{p1[:,0].min():.3f}, {p1[:,0].max():.3f}] '
      f'y [{p1[:,1].min():.3f}, {p1[:,1].max():.3f}] '
      f'z [{p1[:,2].min():.3f}, {p1[:,2].max():.3f}]')

# 3) Voxel-density uniformity - the test that surfaced the void-clustering
# issue. Bin rod midpoints into a 4x4x4 grid (cells of side ~2.86 µm at
# BOX=11.44) and measure spread + boundary-vs-interior bias. The 1µm
# coverage check above is too fine to see this - at 1µm there are ~3
# midpoints/cell, so empty cells dominate either way. The 4³ grid has
# ~26 midpoints expected per cell.
#
# Reference example (lsu_example_ends.txt, BOX=11.44):
#    4³ voxels:  std=3.65  min=17  max=35   corner sum=221  centre sum=183
#                                           corner/centre ratio=1.21
# Pre-fix run (lsu_generated_4.txt, BOX=11.44, relax_global_every=1000):
#    4³ voxels:  std=9.72  min=7   max=55   corner sum=222  centre sum=233
#                                           corner/centre ratio=0.95 (flipped)
# Pass criteria for the gated-fallback fix: std <= 4.0 AND
# corner/centre ratio in [1.0, 1.4].
midpts = (p1 + p2) / 2.0
midpts_canon = midpts - BOX * np.round(midpts / BOX)
vbins = np.linspace(-half, half, 5)
Hv, _ = np.histogramdd(midpts_canon, bins=(vbins, vbins, vbins))
# Boundary mask: any voxel with at least one index in {0, 3} along any axis.
bdry_mask = np.zeros(Hv.shape, dtype=bool)
bdry_mask[0, :, :] = bdry_mask[-1, :, :] = True
bdry_mask[:, 0, :] = bdry_mask[:, -1, :] = True
bdry_mask[:, :, 0] = bdry_mask[:, :, -1] = True
# Corner mask: 8 voxels with indices all in {0, 3}.
corner_mask = np.zeros(Hv.shape, dtype=bool)
for ix in (0, -1):
    for iy in (0, -1):
        for iz in (0, -1):
            corner_mask[ix, iy, iz] = True
# Centre mask: 8 voxels with indices all in {1, 2} (the inner 2x2x2).
centre_mask = np.zeros(Hv.shape, dtype=bool)
centre_mask[1:3, 1:3, 1:3] = True
expected = midpts_canon.shape[0] / Hv.size
print(f'4³ voxel midpoints (cell ~{BOX/4:.2f} µm, expected {expected:.1f}/cell):')
print(f'  spread     mean={Hv.mean():.2f} std={Hv.std():.2f} '
      f'min={Hv.min():.0f} max={Hv.max():.0f}')
print(f'  bdry/int   bdry mean={Hv[bdry_mask].mean():.2f}  '
      f'int mean={Hv[~bdry_mask].mean():.2f}')
print(f'  corner/centre  corner sum={Hv[corner_mask].sum():.0f}  '
      f'centre sum={Hv[centre_mask].sum():.0f}  '
      f'ratio={Hv[corner_mask].sum() / max(Hv[centre_mask].sum(), 1):.2f}')
print(f'  ref target std=3.65 corner/centre=1.21  '
      f'(pass: std<=4.0 AND ratio in [1.0, 1.4])')

rod count : 1645
lengths   : mean=0.809 std=0.025
  quartiles  min=0.730  5%=0.768  Q1=0.793  med=0.810  Q3=0.826  95%=0.850  max=0.889
  ref target  mean=0.800 std=0.029 (reach with enough WWW iters)
1 µm³ vertex coverage: 1728 cells, 980 empty (56.7%)  (reference: 54.5%)
  largest empty clusters: [941, 9, 5, 2, 2]  (big single cluster is normal at this density due to PBC percolation; what was *wrong* before was a big cluster on a box face)
box span   : x [-5.682, 5.715] y [-5.673, 5.720] z [-5.702, 5.698]
4³ voxel midpoints (cell ~2.86 µm, expected 25.7/cell):
  spread     mean=25.70 std=8.84 min=3 max=49
  bdry/int   bdry mean=26.09  int mean=23.00
  corner/centre  corner sum=241  centre sum=184  ratio=1.31
  ref target std=3.65 corner/centre=1.21  (pass: std<=4.0 AND ratio in [1.0, 1.4])
